In [1]:
import os
import re
import shutil
from bs4 import BeautifulSoup

# Input: tree of files
IN_ROOT = "C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wanttoknow"
# Output: flat directory (already created)
OUT_DIR = "C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat"   # set to "." if you really want them in wtktxt directly

# Extensions we care about
EXTS = (".html", ".shtml", ".php")

os.makedirs(OUT_DIR, exist_ok=True)

def file_to_text(path):
    """Read file, strip PHP blocks and HTML tags, return visible text."""
    with open(path, "rb") as f:
        raw = f.read()
    html = raw.decode("utf-8", errors="ignore")

    # Strip PHP blocks to avoid confusing the HTML parser
    html = re.sub(r"<\?php.*?\?>", "", html, flags=re.DOTALL | re.IGNORECASE)

    soup = BeautifulSoup(html, "lxml")

    # Remove obvious non-content
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(" ", strip=True)
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)
    return text

def sentences_from_text(text):
    """Naively split text into sentences."""
    # Split on . ! ? followed by whitespace
    parts = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in parts if s.strip()]

def is_english_like(sentence):
    """Heuristic: looks like an English sentence."""
    # Must contain ASCII letters
    if not re.search(r"[A-Za-z]", sentence):
        return False
    # Must have at least a few words
    if len(sentence.split()) < 4:
        return False
    # Must be at least some characters long
    if len(sentence) < 20:
        return False
    return True

def has_four_english_sentences_in_a_row(text):
    """Return True if there are >=4 English-like sentences consecutively."""
    sents = sentences_from_text(text)
    run = 0
    for s in sents:
        if is_english_like(s):
            run += 1
            if run >= 4:
                return True
        else:
            run = 0
    return False

def main():
    kept = 0
    checked = 0

    for root, dirs, files in os.walk(IN_ROOT):
        for name in files:
            if not name.lower().endswith(EXTS):
                continue

            full_path = os.path.join(root, name)
            rel_path = os.path.relpath(full_path, IN_ROOT)
            checked += 1

            try:
                text = file_to_text(full_path)
            except Exception as e:
                print(f"[WARN] Skipping {rel_path}: read error {e}")
                continue

            if not has_four_english_sentences_in_a_row(text):
                # Does not meet the content threshold
                continue

            # Flatten: encode the original path into the filename
            # to avoid name collisions (same filename in different dirs)
            flat_name = rel_path.replace(os.sep, "__")
            dest_path = os.path.join(OUT_DIR, flat_name)

            shutil.copy2(full_path, dest_path)
            kept += 1
            print(f"[KEEP] {rel_path} -> {dest_path}")

    print(f"\nChecked {checked} files, kept {kept} with >=4 English sentences in a row.")
    
main()    

[KEEP] 0000000header amberCOVIDcopy.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\0000000header amberCOVIDcopy.shtml
[KEEP] 0000000header-both.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\0000000header-both.shtml
[KEEP] 0000000header.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\0000000header.shtml
[KEEP] 0000000headerAmberCopy.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\0000000headerAmberCopy.shtml
[KEEP] 0000000quoteoftheweek.php -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\0000000quoteoftheweek.php
[KEEP] 000000box911.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\000000box911.shtml
[KEEP] 000000boxbanking.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\000000boxbanking.shtml
[KEEP] 000000boxcoronavirus.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\000000boxcoronavirus.shtml
[KEEP] 0

[KEEP] 011218reuters.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\011218reuters.shtml
[KEEP] 011225nytimes.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\011225nytimes.shtml
[KEEP] 011230nytimes.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\011230nytimes.shtml
[KEEP] 020100fe.911investigationworldtradecenterwtccollapse.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\020100fe.911investigationworldtradecenterwtccollapse.shtml
[KEEP] 020110independent.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\020110independent.shtml
[KEEP] 020114johnoneillfbi911wtcsecurity.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\020114johnoneillfbi911wtcsecurity.shtml
[KEEP] 020204newsweek.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\020204newsweek.shtml
[KEEP] 020213londontimes.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtk

[KEEP] 050127mindcontrolarticle.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\050127mindcontrolarticle.shtml
[KEEP] 050129fredburkscelebrity.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\050129fredburkscelebrity.shtml
[KEEP] 050131sibeledmonds.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\050131sibeledmonds.shtml
[KEEP] 050203cianazilink.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\050203cianazilink.shtml
[KEEP] 050205willpoweryieldsforests.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\050205willpoweryieldsforests.shtml
[KEEP] 050207kevinshelleysresignation.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\050207kevinshelleysresignation.shtml
[KEEP] 050209energyscandals.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\050209energyscandals.shtml
[KEEP] 050211momentoflove.shtml -> C:/Users/fixin/Desktop/PEERSwork/b

[KEEP] 050718karlrovevalerieplame.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\050718karlrovevalerieplame.shtml
[KEEP] 050720heartmath.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\050720heartmath.shtml
[KEEP] 050723robertmcnamaraapocalypse.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\050723robertmcnamaraapocalypse.shtml
[KEEP] 050726scandal911.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\050726scandal911.shtml
[KEEP] 050729cancercure.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\050729cancercure.shtml
[KEEP] 050802coverupnewssummary.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\050802coverupnewssummary.shtml
[KEEP] 050806dieboldcoverup.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\050806dieboldcoverup.shtml
[KEEP] 050809profoundspiritualwriting.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt

[KEEP] 060324cnnquestions911.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\060324cnnquestions911.shtml
[KEEP] 060327newsarticles.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\060327newsarticles.shtml
[KEEP] 060330wanttoknow.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\060330wanttoknow.shtml
[KEEP] 060403newsarticles.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\060403newsarticles.shtml
[KEEP] 060407bestdocumentary9-11.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\060407bestdocumentary9-11.shtml
[KEEP] 060411gaousgovernmentfinances.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\060411gaousgovernmentfinances.shtml
[KEEP] 060416newsarticles.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\060416newsarticles.shtml
[KEEP] 060421inspirationaltransformation.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/

...

[KEEP] mind_control\scientology_remote_viewing.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\mind_control__scientology_remote_viewing.shtml
[KEEP] mind_control\cia_mind_control_documents_orig\index.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\mind_control__cia_mind_control_documents_orig__index.shtml
[KEEP] mind_control\cia_mind_control_documents_orig_TIF\index.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\mind_control__cia_mind_control_documents_orig_TIF__index.shtml
[KEEP] mind_control\foia_mind_control\144700.3_project_artichoke_sodium_pentothal_amytal.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\mind_control__foia_mind_control__144700.3_project_artichoke_sodium_pentothal_amytal.shtml
[KEEP] mind_control\foia_mind_control\17395_drugs_electric_shock_mind_control.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\mind_control__foia_mind_control__17395_drugs_el

[KEEP] mk\trauma-based-mind-control.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\mk__trauma-based-mind-control.shtml
[KEEP] nde\journey-of-souls-life-between-lives.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\nde__journey-of-souls-life-between-lives.shtml
[KEEP] nde\near-death-account-nde.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\nde__near-death-account-nde.shtml
[KEEP] nde\near-death-experience-lessons.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\nde__near-death-experience-lessons.shtml
[KEEP] nde\near-death-experiences-ndes.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\nde__near-death-experiences-ndes.shtml
[KEEP] nde\past-lives-articles-news.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\nde__past-lives-articles-news.shtml
[KEEP] nde\past_lives_children.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat

[KEEP] ufos\robert_salas_ufo.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\ufos__robert_salas_ufo.shtml
[KEEP] ufos\ufo-center-announcement.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\ufos__ufo-center-announcement.shtml
[KEEP] ufos\ufo-disclosure-breakthrough-tech-human-consciousness.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\ufos__ufo-disclosure-breakthrough-tech-human-consciousness.shtml
[KEEP] ufos\ufo-disclosure-danny-sheehan-interview.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\ufos__ufo-disclosure-danny-sheehan-interview.shtml
[KEEP] ufos\ufo-leak-of-century.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\ufos__ufo-leak-of-century.shtml
[KEEP] ufos\ufo-news.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\ufos__ufo-news.shtml
[KEEP] ufos\ufos-cia-air-force-new-york-times.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/w

[KEEP] wingmakersorig\acio\www.wingmakers.com\arrow\acio\stevens2.html -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\wingmakersorig__acio__www.wingmakers.com__arrow__acio__stevens2.html
[KEEP] wingmakersorig\acio\www.wingmakers.com\arrow\acio\stevens2.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\wingmakersorig__acio__www.wingmakers.com__arrow__acio__stevens2.shtml
[KEEP] wingmakersorig\wingmakers\WingMakersindex2.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\wingmakersorig__wingmakers__WingMakersindex2.shtml
[KEEP] wingmakersorig\wingmakers\WingMakersindex3.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\wingmakersorig__wingmakers__WingMakersindex3.shtml
[KEEP] wingmakersorig\wingmakers\WingMakersindex2_files\intro.html -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\wingmakersorig__wingmakers__WingMakersindex2_files__intro.html
[KEEP] wingmakersorig\wingmakers\WingMakersi

[KEEP] wingmakersorig\www.wingmakers.com\arrow\chambers\poetry\poem23.html -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\wingmakersorig__www.wingmakers.com__arrow__chambers__poetry__poem23.html
[KEEP] wingmakersorig\www.wingmakers.com\arrow\chambers\poetry\poem23.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\wingmakersorig__www.wingmakers.com__arrow__chambers__poetry__poem23.shtml
[KEEP] wingmakersorig\www.wingmakers.com\arrow\chambers\poetry\poem3.html -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\wingmakersorig__www.wingmakers.com__arrow__chambers__poetry__poem3.html
[KEEP] wingmakersorig\www.wingmakers.com\arrow\chambers\poetry\poem3.shtml -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\wingmakersorig__www.wingmakers.com__arrow__chambers__poetry__poem3.shtml
[KEEP] wingmakersorig\www.wingmakers.com\arrow\chambers\poetry\poem4.html -> C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat\wing

In [1]:
import os
import re
import csv
import json
from bs4 import BeautifulSoup
from bs4.element import Tag

# Directory that holds all your flattened HTML/PHP/SHTML files
IN_DIR = "C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/wtktxt_flat"
# Output directory for cleaned text files
CLEAN_DIR = "C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/clean_text"
os.makedirs(CLEAN_DIR, exist_ok=True)

LINKS_CSV = "wtk_links.csv"
LINKS_JSONL = "wtk_links.jsonl"

EXTS = (".html", ".shtml", ".php")

def read_html(path):
    with open(path, "rb") as f:
        raw = f.read()
    html = raw.decode("utf-8", errors="ignore")
    # Strip PHP blocks
    html = re.sub(r"<\?php.*?\?>", "", html, flags=re.DOTALL | re.IGNORECASE)
    return html

# -------- Boilerplate removal (header/footer/nav/sidebar) --------

BOILERPLATE_SELECTORS = [
    "nav",
    "header",
    "footer",
    "aside",
    "div#header",
    "div#footer",
    "div#sidebar",
    "div#leftcol",
    "div#rightcol",
    "td#leftcol",
    "td#rightcol",
    "table#nav",
    "div.nav",
    "div.menu",
    "ul#nav",
    "ul.menu",
    ".navbar",
    ".sidebar",
    ".footer",
    ".topnav",
]

ID_CLASS_KEYWORDS = [
    "nav",
    "menu",
    "header",
    "footer",
    "sidebar",
    "sidecol",
    "leftcol",
    "rightcol",
    "masthead",
    "banner",
]

def remove_boilerplate(soup: BeautifulSoup):
    # Remove by explicit selectors
    for sel in BOILERPLATE_SELECTORS:
        for tag in soup.select(sel):
            tag.decompose()

    # Remove tags whose id/class *contains* known boilerplate keywords
    for tag in list(soup.find_all(True)):
        # Be paranoid: only handle real Tag objects
        if not isinstance(tag, Tag):
            continue

        # Use attrs dict so we never call .get on None
        attrs = getattr(tag, "attrs", {}) or {}
        id_attr = attrs.get("id", "") or ""
        class_attr = attrs.get("class", []) or []

        if isinstance(class_attr, str):
            class_str = class_attr
        else:
            class_str = " ".join(class_attr)

        haystack = f"{id_attr} {class_str}".lower()
        if any(kw in haystack for kw in ID_CLASS_KEYWORDS):
            tag.decompose()

# -------- Link handling: inline Markdown + metadata collection --------

def convert_links_to_markdown_and_collect(soup):
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        anchor_text = " ".join(a.get_text(strip=True).split())
        if not anchor_text:
            anchor_text = href
        links.append({"href": href, "text": anchor_text})
        markdown = f"[{anchor_text}]({href})"
        a.replace_with(markdown)
    return links

# -------- Text extraction with paragraph/line preservation --------

def apply_inline_markdown_formatting(soup):
    """
    Turn <i>/<em> into *...* and <b>/<strong> into **...** in the DOM,
    so soup.get_text() will preserve the markdown markers.
    """
    # italics/emphasis → *...*
    for tag in soup.find_all(["i", "em"]):
        tag.insert_before("*")
        tag.insert_after("*")
        tag.unwrap()  # remove the tag but keep its contents

    # bold/strong → **...**
    for tag in soup.find_all(["b", "strong"]):
        tag.insert_before("**")
        tag.insert_after("**")
        tag.unwrap()

def html_to_preserved_markdown_text(html):
    soup = BeautifulSoup(html, "lxml")

    # Remove scripts/styles early
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    # Boilerplate removal – keep your existing function
    try:
        remove_boilerplate(soup)
    except Exception as e:
        print(f"[WARN] Boilerplate removal issue, continuing: {e}")

    # NEW: convert <i>/<em>/<b>/<strong> to markdown markers
    apply_inline_markdown_formatting(soup)

    # Convert links inline to [text](url) and collect metadata
    links = convert_links_to_markdown_and_collect(soup)

    # <br> -> newline
    for br in soup.find_all("br"):
        br.replace_with("\n")

    # Add blank lines around block elements
    block_tags = (
        "p", "div", "section", "article",
        "ul", "ol", "li",
        "header", "footer", "nav",
        "h1","h2","h3","h4","h5","h6",
        "blockquote", "pre"
    )
    for tag in soup.find_all(block_tags):
        tag.insert_before("\n")
        tag.insert_after("\n")

    # Now flatten to text (markdown markers and [links](url) remain)
    text = soup.get_text()

    # Normalize blank lines (max 2 in a row)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)
    text = "\n".join(line.rstrip() for line in text.splitlines())
    text = text.strip()

    return text, links

# -------- Main loop --------

all_links_rows = []
all_links_json = []
processed = 0

for name in os.listdir(IN_DIR):
    if not name.lower().endswith(EXTS):
        continue

    src_path = os.path.join(IN_DIR, name)

    try:
        html = read_html(src_path)
        text, links = html_to_preserved_markdown_text(html)
    except Exception as e:
        print(f"[WARN] Skipping {name}: {e}")
        continue

    txt_name = re.sub(r"\.(html|shtml|php)$", ".txt", name, flags=re.IGNORECASE)
    out_path = os.path.join(CLEAN_DIR, txt_name)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)

    for link in links:
        all_links_rows.append([name, link["href"], link["text"]])
        all_links_json.append({
            "source_file": name,
            "href": link["href"],
            "text": link["text"],
        })

    processed += 1
    if processed % 200 == 0:
        print(f"Processed {processed} files...")

print(f"Done. Processed {processed} files.")
print(f"Clean text files in: {CLEAN_DIR}/")
print(f"Total links captured: {len(all_links_rows)}")

with open(LINKS_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["source_file", "href", "anchor_text"])
    writer.writerows(all_links_rows)

with open(LINKS_JSONL, "w", encoding="utf-8") as f:
    for rec in all_links_json:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Wrote: {LINKS_CSV}, {LINKS_JSONL}")


Processed 200 files...
Processed 400 files...
Processed 600 files...
Processed 800 files...
Processed 1000 files...
Processed 1200 files...
Processed 1400 files...
Processed 1600 files...
Processed 1800 files...
Processed 2000 files...
Processed 2200 files...
Processed 2400 files...
Processed 2600 files...
Processed 2800 files...
Processed 3000 files...
Processed 3200 files...
Processed 3400 files...
Done. Processed 3514 files.
Clean text files in: C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/clean_text/
Total links captured: 141427
Wrote: wtk_links.csv, wtk_links.jsonl


In [2]:
import os
import re

INPUT_DIR = "C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/clean_text"           # where the current cleaned text is
OUTPUT_DIR = "C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/clean_text_final"                 # where to write fully sanitized text
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Regex for removing all non-printing characters:
#   - \x00–\x1F control chars
#   - \x7F delete
#   - preserves normal ASCII, UTF-8 letters, punctuation, emojis, etc.
NONPRINTING = re.compile(r"[\x00-\x1F\x7F]")

for fname in os.listdir(INPUT_DIR):
    if not fname.lower().endswith(".txt"):
        continue

    in_path = os.path.join(INPUT_DIR, fname)
    out_path = os.path.join(OUTPUT_DIR, fname)

    with open(in_path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    # ---------- Sanitization steps ----------

    # 1. Remove all indentation (leading whitespace on each line)
    lines = [re.sub(r"^[ \t]+", "", line) for line in text.splitlines()]

    # 2. Remove non-printing characters
    lines = [NONPRINTING.sub("", line) for line in lines]

    # 3. (Optional) Collapse multiple internal spaces
    #    Comment out if you want exact spacing.
    # lines = [re.sub(r"\s{2,}", " ", line) for line in lines]

    # 4. Re-join into final text
    cleaned = "\n".join(lines).rstrip()

    # ----------------------------------------

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(cleaned)

print("Done. Sanitized text files written to:", OUTPUT_DIR)


Done. Sanitized text files written to: C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/clean_text_final


In [9]:
import os

print("CWD:", os.getcwd())

base_dir = r"C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/clean_text_final"
print("Target base_dir:", base_dir)

print("\nEntries in base_dir:")
for name in os.listdir(base_dir):
    print("  ", name)

txt_files = [f for f in os.listdir(base_dir) if f.lower().endswith(".txt")]
print(f"\nNumber of .txt files directly in base_dir: {len(txt_files)}")


CWD: C:\Users\fixin
Target base_dir: C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/clean_text_final

Entries in base_dir:
   00-box__000000box911.txt
   00-box__Amber__220906.txt
   00-box__Amber__220913.txt
   00-box__Amber__220920.txt
   00-box__Amber__AmberNewsSummaryTemplate.txt
   00-box__Amber__AmberTemplate.txt
   00-templates__0-template-sections.txt
   00-templates__00000wtkquote.txt
   00-templates__0000inspwtk.txt
   00-templates__00newstemplate.txt
   00-templates__0egextext.txt
   00-templates__0newtemplatepaidarchive.txt
   00-templates__0paymentrequird.txt
   00-templates__0redirect.txt
   00-templates__0templatearticleonly.txt
   00-templates__0templatedisappeared.txt
   00-templates__0templatedisappearednon911.txt
   00-templates__wtk-article-template.txt
   0000000header amberCOVIDcopy.txt
   0000000header-both.txt
   0000000header.txt
   0000000headerAmberCopy.txt
   0000000quoteoftheweek.txt
   000000box911.txt
   000000boxbanking.txt
   000000boxcoronavirus.tx

   22__7-forever-chemicals-drinking-water.txt
   22__7-roundup-cancer-lawsuits-proceed.txt
   22__7-sexual-predator-fbi.txt
   22__8-1-30-kids-autism.txt
   22__8-china-surveillance-state.txt
   22__8-john-bolton-admits-planning-coups.txt
   22__8-saudi-prison-tweets.txt
   22__8-suppression-free-speech-collusion.txt
   22__9-companies-inflation-hike-prices.txt
   22__9-covid-boosters-authorized-without-human-trials.txt
   22__9-lockdowns-erased-progress-math-reading.txt
   22__9-pentagon-social-media-manipulations.txt
   23__1-artificial-intelligence-in-space-warfare.txt
   23__1-microplastics-in-unborn-babies.txt
   23__1-overcounting-covid-deaths.txt
   23__1-ufo-reports-skyrocket.txt
   23__1-ufo-rockets-skyrocket.txt
   23__1-wells-fargo-fined-billions.txt
   23__10-civilian-deaths-in-gaza.txt
   23__10-israels-surprising-intelligence-failure.txt
   23__10-jews-and-arabs-join-forces.txt
   23__10-nasa-new-ufo-czar.txt
   23__10-teen-boys-targeted-by-sexual-blackmailers.txt
   23__

   community__sites__all__themes__bootstrap__templates__bootstrap__bootstrap-dropdown.vars.txt
   community__sites__all__themes__bootstrap__templates__bootstrap__bootstrap-panel.vars.txt
   community__sites__all__themes__bootstrap__templates__file__file-managed-file.func.txt
   community__sites__all__themes__bootstrap__templates__file__file-widget-multiple.func.txt
   community__sites__all__themes__bootstrap__templates__file__file-widget.func.txt
   community__sites__all__themes__bootstrap__templates__filter__filter-tips.func.txt
   community__sites__all__themes__bootstrap__templates__menu__menu-link.func.txt
   community__sites__all__themes__bootstrap__templates__menu__menu-local-task.func.txt
   community__sites__all__themes__bootstrap__templates__system__container.func.txt
   community__sites__all__themes__bootstrap__templates__system__form-element-label.func.txt
   community__sites__all__themes__bootstrap__templates__system__form-element.func.txt
   community__sites__all__themes__b

   nesara.txt
   newenergyinformation.txt
   newenergysources.txt
   newpearlharbor.txt
   nickberg.txt
   nickberg911.txt
   nikolateslamagnifyingtransmitter.txt
   noisegun.txt
   norad911drills.txt
   northwoods.txt
   officialsquestion911commissionreport.txt
   officialstestifyufos.txt
   operationnorthwoods.txt
   overwhelmed.txt
   pammonday.txt
   pandorasbox.txt
   pandorasboxcv.txt
   peacetogether.txt
   pearlharborradio.txt
   permanentmagnets.txt
   philosophicalessay.txt
   plottoseizethewhitehouse.txt
   pollfraud.txt
   postelectronicvotingproblems.txt
   powercorrupts.txt
   poweroflove.txt
   powerofnightmares.txt
   presidentialresponses.txt
   presidentialsurprises.txt
   psi__psi-paranormal-research.txt
   qualityoflife.txt
   Quotes__quotes-for-better-life.txt
   Quotes__quotes-wingmakers.txt
   radiantenergy.txt
   raymcgoverncia.txt
   realitiesofwar.txt
   redout.txt
   redout2.txt
   redoutemail.txt
   redoutthankyou.txt
   remoteviewing__770807-psychic-spying-

In [3]:
import os
import re
import unicodedata

BASE_DIR = r"C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/clean_text_final"

# control chars except newline (we keep \n)
CONTROL_CHARS = re.compile(r'[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F]')

def clean_text(text: str) -> str:
    # 1. Unicode normalization
    t = unicodedata.normalize("NFKC", text)

    # 2. Standardize line endings
    t = t.replace("\r\n", "\n").replace("\r", "\n")

    # 3. Replace non-breaking / narrow spaces with normal space
    NBSP_LIKE = ["\u00A0", "\u2007", "\u202F"]
    for ch in NBSP_LIKE:
        t = t.replace(ch, " ")

    # 4. Curly quotes / apostrophes -> straight
    replacements = {
        "‘": "'", "’": "'", "‚": "'", "‛": "'", "´": "'", "ʹ": "'", "ʻ": "'",
        "“": '"', "”": '"', "„": '"', "‟": '"', "«": '"', "»": '"',
    }
    for src, dst in replacements.items():
        t = t.replace(src, dst)

    # 5. Normalize dashes
    t = t.replace("\u2013", "-")  # en dash
    t = t.replace("\u2014", "-")  # em dash
    t = t.replace("\u2212", "-")  # minus sign

    # 6. Remove remaining control chars (keep newlines)
    t = CONTROL_CHARS.sub("", t)

    # 7. Per-line whitespace cleanup
    lines = []
    for line in t.split("\n"):
        # strip leading/trailing spaces & tabs
        line = line.strip()
        # collapse multiple internal spaces/tabs to a single space
        line = re.sub(r"[ \t]{2,}", " ", line)
        lines.append(line)

    t = "\n".join(lines)

    # 8. Limit consecutive blank lines to max 2
    t = re.sub(r"\n{3,}", "\n\n", t)

    # 9. Final trim
    return t.strip()

processed = 0
for root, dirs, files in os.walk(BASE_DIR):
    for fname in files:
        if not fname.lower().endswith(".txt"):
            continue

        path = os.path.join(root, fname)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            original = f.read()

        cleaned = clean_text(original)

        with open(path, "w", encoding="utf-8") as f:
            f.write(cleaned)

        processed += 1
        if processed % 200 == 0:
            print(f"Cleaned {processed} files so far...")

print(f"Done. Cleaned {processed} .txt files under {BASE_DIR}.")


Cleaned 200 files so far...
Cleaned 400 files so far...
Cleaned 600 files so far...
Cleaned 800 files so far...
Cleaned 1000 files so far...
Cleaned 1200 files so far...
Cleaned 1400 files so far...
Cleaned 1600 files so far...
Cleaned 1800 files so far...
Cleaned 2000 files so far...
Cleaned 2200 files so far...
Cleaned 2400 files so far...
Cleaned 2600 files so far...
Cleaned 2800 files so far...
Cleaned 3000 files so far...
Cleaned 3200 files so far...
Cleaned 3400 files so far...
Done. Cleaned 3480 .txt files under C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/clean_text_final.


In [4]:
import os
import re
import json
import html
import unicodedata
from pathlib import Path

# Path to your JSON file

json_path = Path(r"C:\datasources\ai\articles_df.json")

print("JSON exists:", json_path.exists())

with json_path.open("r", encoding="utf-8") as f:
    articles = json.load(f)

len(articles)


JSON exists: True


13851

In [5]:
def normalize_for_match(s: str) -> str:
    if not s:
        return ""
    # HTML unescape & strip tags if any
    s = html.unescape(s)

    # Strip HTML tags
    s = re.sub(r"<[^>]+>", " ", s)

    # Unicode normalize
    s = unicodedata.normalize("NFKC", s)

    # Lowercase
    s = s.lower()

    # Replace NBSP-like spaces
    for ch in ["\u00a0", "\u2007", "\u202f"]:
        s = s.replace(ch, " ")

    # Normalize quotes + dashes (roughly matching what you did earlier)
    replacements = {
        "‘": "'", "’": "'", "‚": "'", "‛": "'", "´": "'", "ʹ": "'", "ʻ": "'",
        "“": '"', "”": '"', "„": '"', "‟": '"', "«": '"', "»": '"',
        "\u2013": "-", "\u2014": "-", "\u2212": "-",
    }
    for src, dst in replacements.items():
        s = s.replace(src, dst)

    # Turn all whitespace (including newlines) into single spaces
    s = re.sub(r"\s+", " ", s).strip()

    return s

summary_probes = []

for rec in articles:
    # Choose which field to use as the summary source
    raw = rec.get("Excerpt") or rec.get("__text_for_tfidf__") or ""
    norm = normalize_for_match(raw)
    if len(norm) < 80:
        continue  # too short to be a reliable match

    probe = norm[:200]  # first 200 chars used as the "needle"
    summary_probes.append((rec.get("ID"), probe))

len(summary_probes)


13851

In [7]:
import shutil

# Base directory for cleaned text
BASE_DIR = Path("C:/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/clean_text_final")

# Where to move duplicates
DUP_DIR = BASE_DIR / "duplicates"
DUP_DIR.mkdir(exist_ok=True)

def file_is_duplicate(txt_path: Path, probes) -> tuple[bool, list]:
    """Return (is_duplicate, matching_ids) for this file."""
    with txt_path.open("r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()

    norm = normalize_for_match(raw)
    matching_ids = []

    for art_id, probe in probes:
        if probe and probe in norm:
            matching_ids.append(art_id)
            # If you only care about a yes/no answer,
            # you could break here for speed.
            # break

    return (len(matching_ids) > 0, matching_ids)

duplicates = []

for root, dirs, files in os.walk(BASE_DIR):
    root_path = Path(root)
    # don't dive into the duplicates folder itself if you rerun
    if root_path == DUP_DIR:
        continue

    for fname in files:
        if not fname.lower().endswith(".txt"):
            continue

        fpath = root_path / fname

        is_dup, ids = file_is_duplicate(fpath, summary_probes)
        if is_dup:
            # Move file to duplicates dir, preserving filename
            dest = DUP_DIR / fpath.name
            # If you're worried about name collisions, adjust here
            if dest.exists():
                # Add a suffix if already exists
                dest = DUP_DIR / f"{fpath.stem}__dup{fpath.suffix}"

            shutil.move(str(fpath), str(dest))
            duplicates.append((str(fpath), ids))
            print(f"[DUP] {fpath} -> {dest} | matched IDs: {ids}")

print(f"\nTotal duplicate files moved: {len(duplicates)}")


[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\00-box__Amber__220906.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\00-box__Amber__220906.txt | matched IDs: [12078, 12079, 12081, 12082, 12086, 12087]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\00-box__Amber__220913.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\00-box__Amber__220913.txt | matched IDs: [12091, 12092, 12093, 12095, 12097, 12098]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\00-box__Amber__220920.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\00-box__Amber__220920.txt | matched IDs: [12101, 12102, 12103, 12104, 12105, 12106, 12108, 12110]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\00-box__Amber__AmberNewsSummaryTemplate.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final

[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\008__080515vaticanextraterrestrialsiraqcorruptionnanoparticlethreats.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\008__080515vaticanextraterrestrialsiraqcorruptionnanoparticlethreats.txt | matched IDs: [1818, 1819, 1823, 1824, 1825, 1826, 1827]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\008__080518_dolphins_whales_inspiration_danger.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\008__080518_dolphins_whales_inspiration_danger.txt | matched IDs: [472, 546, 970, 971, 1151, 1196, 1558, 1729, 1772]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\008__080522_government_secrecy_friendly-fire_cover-up_warming.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\008__080522_government_secrecy_friendly-fire_cover-up_warming.txt | matched IDs: [1828, 1829, 

[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\008__081010_guantanamo_detainees_fema_trailers_aig_execs.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\008__081010_guantanamo_detainees_fema_trailers_aig_execs.txt | matched IDs: [2047, 2048, 2050, 2052, 2054, 2055]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\008__081014_elections_irregularities.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\008__081014_elections_irregularities.txt | matched IDs: [565, 860, 875, 1636, 1871, 2007, 2020]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\008__081017_nsa_eavesdropping_fda_corruption_nonlethal_weapons.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\008__081017_nsa_eavesdropping_fda_corruption_nonlethal_weapons.txt | matched IDs: [2057, 2058, 2059, 2060, 2061, 2062, 2063, 2064, 2066, 2067]
[DUP] C:\Us

[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\009__090310_derivatives_danger_pension_risk_oil_storage.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\009__090310_derivatives_danger_pension_risk_oil_storage.txt | matched IDs: [2294, 2295, 2296, 2297, 2298, 2299, 2300, 2301, 2302]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\009__090317_aig_bonuses_guantanamo_unchanged_credit_crunch.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\009__090317_aig_bonuses_guantanamo_unchanged_credit_crunch.txt | matched IDs: [1178, 2304, 2306, 2308, 2309, 2310, 2312, 2314]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\009__090323_electoral_fraud_aig_scandal_allergies_cured.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\009__090323_electoral_fraud_aig_scandal_allergies_cured.txt | matched IDs: [2316, 2317, 2318

[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\009__090817_health_industry_bailout_funds_computers_claimed.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\009__090817_health_industry_bailout_funds_computers_claimed.txt | matched IDs: [2558, 2559, 2560, 2563, 2565, 2566, 2567, 2569, 2570]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\009__090824_cia_mock_executions_assassinations_tiring_war.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\009__090824_cia_mock_executions_assassinations_tiring_war.txt | matched IDs: [2571, 2572, 2575, 2576, 2577, 2578, 2579, 2580, 2581, 2582]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\009__090831_moon_rock_fake_swine_flu_cia_torture.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\009__090831_moon_rock_fake_swine_flu_cia_torture.txt | matched IDs: [2584, 2585, 

...

[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\25__4-neo-nazi-terrorist-worked-for-fbi.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\25__4-neo-nazi-terrorist-worked-for-fbi.txt | matched IDs: [13076, 13820, 13822, 13823, 13824, 13825, 13828, 13831]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\25__4-palantir-era-ai-warfare.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\25__4-palantir-era-ai-warfare.txt | matched IDs: [13845, 13846, 13847, 13848, 13850, 13853, 13854, 13855, 13856]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\25__4-parkinsons-man-made-disease.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\25__4-parkinsons-man-made-disease.txt | matched IDs: [13859, 13860, 13864, 13868, 13869]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\25__5-ai-powered-underco

[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\articles__parallel-government-870705.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\articles__parallel-government-870705.txt | matched IDs: [7471]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\atomicbombcoverup.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\atomicbombcoverup.txt | matched IDs: [296]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\avianflu.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\avianflu.txt | matched IDs: [71, 83, 107, 166, 199, 203, 205, 221]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\banking_finance__derivatives_market_bubble_financial.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\banking_finance__derivatives_market_bubble_financial.txt | matched IDs

[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\g__private-prisons-corruption-news-articles.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\g__private-prisons-corruption-news-articles.txt | matched IDs: [2299, 2454, 3764, 4368, 4686, 4702, 4751, 5230, 6228, 6504, 7030, 7182, 7366, 8401, 9376, 9443]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\g__question_authority_question_reality.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\g__question_authority_question_reality.txt | matched IDs: [19, 1554, 879, 896, 1058, 2806, 2858, 2954, 3171, 3401]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\g__rothschild-family.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\g__rothschild-family.txt | matched IDs: [5547, 5548, 5549, 5597, 5598, 5612, 5629, 6042]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wt

[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\h__ebola-pandemic.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\h__ebola-pandemic.txt | matched IDs: [83, 501, 2297, 2419, 2648, 2743, 2820, 3174, 4367, 5207, 6203]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\h__psychedelic-drugs-news-articles.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\h__psychedelic-drugs-news-articles.txt | matched IDs: [4198, 4670, 6888, 7369, 8343, 8409, 8422, 8994, 9044, 9045, 9132, 9437, 9452]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\h__vaccine-articles.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\h__vaccine-articles.txt | matched IDs: [176, 328, 384, 764, 3383, 4353, 4382, 5643, 6335, 6588, 6832]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\h__vaccines-autism-mercury.txt -> C:\

[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\microchipimplants.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\microchipimplants.txt | matched IDs: [423, 210, 400, 401, 999]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\mind_control__behavior_modification.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\mind_control__behavior_modification.txt | matched IDs: [19, 768, 896, 1227, 1058, 1202, 1215, 1295, 3014, 3836, 4618]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\mk__behavior-modification-news-articles.txt -> C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\duplicates\mk__behavior-modification-news-articles.txt | matched IDs: [19, 768, 896, 1227, 1058, 1202, 1215, 1295, 3014, 3836, 4618]
[DUP] C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\mk__bombers-assassins-controlled-cia-government.t

In [8]:
import os
import shutil
from pathlib import Path

# Base dir for cleaned text
BASE = Path(r"C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final")

DUP = BASE / "duplicates"

print("Duplicates folder exists:", DUP.exists())

moved = 0

for item in DUP.iterdir():
    if item.is_file():
        fname = item.name
        if fname and fname[0].isalpha():    # first character is a letter
            dest = BASE / fname

            # Avoid overwriting — optional safeguard
            if dest.exists():
                dest = BASE / f"{item.stem}_from_dup{item.suffix}"

            shutil.move(str(item), str(dest))
            moved += 1
            print(f"[MOVE] {item.name} → {dest.name}")

print(f"\nTotal moved back: {moved}")


Duplicates folder exists: True
[MOVE] abledanger911.txt → abledanger911.txt
[MOVE] articles__parallel-government-870705.txt → articles__parallel-government-870705.txt
[MOVE] atomicbombcoverup.txt → atomicbombcoverup.txt
[MOVE] avianflu.txt → avianflu.txt
[MOVE] banking_finance__derivatives_market_bubble_financial.txt → banking_finance__derivatives_market_bubble_financial.txt
[MOVE] beyondfear__how-consciousness-research-can-heal-a-divided-world.txt → beyondfear__how-consciousness-research-can-heal-a-divided-world.txt
[MOVE] corruptiongovernmentmilitary.txt → corruptiongovernmentmilitary.txt
[MOVE] drugcompanycorruption.txt → drugcompanycorruption.txt
[MOVE] electionscoverups.txt → electionscoverups.txt
[MOVE] electionsmanipulations.txt → electionsmanipulations.txt
[MOVE] electionsproblems.old.txt → electionsproblems.old.txt
[MOVE] electionsproblems.txt → electionsproblems.txt
[MOVE] elections__elections-manipulated.txt → elections__elections-manipulated.txt
[MOVE] elections__elections-

In [9]:
import os
import re
import csv
import json
import html
import unicodedata
from pathlib import Path

# ---------- PATHS (adjust if needed) ----------

# JSON file with article summaries
ARTICLES_JSON = Path(r"C:\datasources\ai\articles_df.json")

# Cleaned text directory
CLEAN_DIR = Path(r"C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final")

# Output CSV (pipe-delimited)
OUT_CSV = CLEAN_DIR / "lora_stage1_dataset.csv"

print("Articles JSON exists:", ARTICLES_JSON.exists())
print("Clean dir exists:", CLEAN_DIR.exists())

# ---------- HELPERS ----------

def sanitize_for_pipe(s: str) -> str:
    """Remove or replace '|' so it doesn't break pipe-delimited CSV."""
    if s is None:
        return ""
    s = str(s)
    # Replace pipe with "I" (or another safe char)
    s = s.replace("|", "I")
    return s

def normalize_whitespace(s: str) -> str:
    """Normalize whitespace; keep newlines, compress internal spaces."""
    # Standardize line endings
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    # Remove non-breaking spaces, etc.
    for ch in ["\u00a0", "\u2007", "\u202f"]:
        s = s.replace(ch, " ")
    # Collapse spaces/tabs per line
    lines = []
    for line in s.split("\n"):
        line = line.strip()
        line = re.sub(r"[ \t]{2,}", " ", line)
        lines.append(line)
    # Limit consecutive blank lines
    s = "\n".join(lines)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def strip_html_preserve_text(s: str) -> str:
    """Turn HTML-y excerpt into plain-ish text."""
    if not s:
        return ""
    # Unescape HTML entities
    s = html.unescape(s)
    # Remove tags
    s = re.sub(r"<[^>]+>", " ", s)
    # Normalize whitespace
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def chunk_text_by_words(text: str, max_words: int = 400, min_last_frac: float = 0.3):
    """
    Chunk text into pieces of up to max_words.
    If the last chunk is very small (< min_last_frac * max_words),
    merge it with the previous chunk.
    """
    words = text.split()
    if not words:
        return []

    chunks = []
    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i+max_words])
        chunks.append(chunk)

    # Merge tiny last chunk
    if len(chunks) > 1:
        last_len = len(chunks[-1].split())
        if last_len < max_words * min_last_frac:
            chunks[-2] = chunks[-2] + " " + chunks[-1]
            chunks.pop()

    return chunks

# ---------- LOAD JSON ARTICLES ----------

with ARTICLES_JSON.open("r", encoding="utf-8") as f:
    articles = json.load(f)

print("Loaded articles:", len(articles))

# ---------- BUILD ROWS ----------

rows = []

# 1) JSON-based entries (keep their original ID)

for rec in articles:
    art_id = rec.get("ID")
    title = sanitize_for_pipe(rec.get("Title", ""))
    date = sanitize_for_pipe(rec.get("Date", ""))
    pub = sanitize_for_pipe(rec.get("Publication", ""))
    excerpt_raw = rec.get("Excerpt") or rec.get("__text_for_tfidf__") or ""

    # Turn HTML excerpt into plain-ish text
    excerpt_text = strip_html_preserve_text(excerpt_raw)
    excerpt_text = normalize_whitespace(excerpt_text)
    excerpt_text = sanitize_for_pipe(excerpt_text)

    # Full display text for chunking
    header_lines = []
    header_lines.append(f"ID: {art_id}")
    if title:
        header_lines.append(f"Title: {title}")
    meta_line_parts = []
    if date:
        meta_line_parts.append(f"Date: {date}")
    if pub:
        meta_line_parts.append(f"Publication: {pub}")
    if meta_line_parts:
        header_lines.append(" | ".join(meta_line_parts))

    header = "\n".join(header_lines).strip()
    # Combine header + excerpt
    combined = header + "\n\n" + excerpt_text if excerpt_text else header

    chunks = chunk_text_by_words(combined, max_words=400)

    for idx, chunk in enumerate(chunks, start=1):
        label = f"JSON_ID_{art_id}_part_{idx}"
        # Label at top of the chunk
        chunk_with_label = f"{label}\n\n{chunk}"

        rows.append({
            "ID": art_id,                      # original ID
            "source": "json_article",
            "orig_id": art_id,
            "filename": "",
            "part": idx,
            "title": title,
            "date": date,
            "publication": pub,
            "text": sanitize_for_pipe(normalize_whitespace(chunk_with_label)),
        })

print("Rows after JSON articles:", len(rows))

# 2) File-based entries from clean_text_final (IDs >= 50001)

next_file_id = 50001

# We'll include only top-level .txt files (change to os.walk if nested)
for root, dirs, files in os.walk(CLEAN_DIR):
    # Skip duplicates folder
    if Path(root) == (CLEAN_DIR / "duplicates"):
        continue

    for fname in sorted(files):
        if not fname.lower().endswith(".txt"):
            continue

        fpath = Path(root) / fname

        with fpath.open("r", encoding="utf-8", errors="ignore") as f:
            text_raw = f.read()

        text_norm = normalize_whitespace(text_raw)
        text_norm = sanitize_for_pipe(text_norm)

        chunks = chunk_text_by_words(text_norm, max_words=400)
        if not chunks:
            continue

        base_id = next_file_id
        next_file_id += 1  # one ID per file (orig_id); chunks get same orig_id but different part

        for idx, chunk in enumerate(chunks, start=1):
            label = f"{fname}_part_{idx}"
            chunk_with_label = f"{label}\n\n{chunk}"

            rows.append({
                "ID": base_id + idx - 1,         # each chunk gets its own unique ID >= 50001
                "source": "file_clean_text",
                "orig_id": base_id,              # same orig_id for all chunks from this file
                "filename": fname,
                "part": idx,
                "title": "",
                "date": "",
                "publication": "",
                "text": sanitize_for_pipe(normalize_whitespace(chunk_with_label)),
            })

print("Total rows after adding files:", len(rows))

# ---------- WRITE PIPE-DELIMITED CSV ----------

fieldnames = [
    "ID",
    "source",
    "orig_id",
    "filename",
    "part",
    "title",
    "date",
    "publication",
    "text",
]

with OUT_CSV.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="|", quoting=csv.QUOTE_MINIMAL)
    writer.writeheader()
    for row in rows:
        writer.writerow(row)

print("Wrote CSV to:", OUT_CSV)
print("Total rows written:", len(rows))


Articles JSON exists: True
Clean dir exists: True
Loaded articles: 13851
Rows after JSON articles: 13858
Total rows after adding files: 27072
Wrote CSV to: C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final\lora_stage1_dataset.csv
Total rows written: 27072


In [10]:
import os
import csv
import json
import time
import math
from pathlib import Path

import requests

BASE_DIR = Path(r"C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final")

INPUT_CSV = BASE_DIR / "lora_stage1_dataset.csv"
OUTPUT_CSV = BASE_DIR / "lora_stage2_qa_pairs.csv"
# Optional JSONL instead/also:
OUTPUT_JSONL = BASE_DIR / "lora_stage2_qa_pairs.jsonl"

print("Input exists:", INPUT_CSV.exists())


Input exists: True


In [11]:
import os
import re
from pathlib import Path

# ---- PATHS ----
BASE_DIR = Path("/mnt/c/Users/fixin/Desktop/PEERSwork/backup2025/wtktxt/clean_text_final")
# Or on Windows-native:
# BASE_DIR = Path(r"C:\Users\fixin\Desktop\PEERSwork\backup2025\wtktxt\clean_text_final")

print("Base dir exists:", BASE_DIR.exists())

# ---- SIZE PRUNING ----

MIN_BYTES = 2048  # delete files <= 2 KB

deleted_small = 0
kept_files = []

for root, dirs, files in os.walk(BASE_DIR):
    for fname in files:
        if not fname.lower().endswith(".txt"):
            continue

        fpath = Path(root) / fname
        size = fpath.stat().st_size

        if size <= MIN_BYTES:
            fpath.unlink()
            deleted_small += 1
        else:
            kept_files.append(fpath)

print(f"Deleted {deleted_small} small files (<= {MIN_BYTES} bytes).")
print(f"Remaining .txt files to clean: {len(kept_files)}")

# ---- HEADER / FOOTER CLEANUP ----

FOOTER_MARKERS = [
    "finding balance: wanttoknow.info inspiration center",
    "see our exceptional archive",
    "please support this important work",
    "explore the mind and heart expanding websites managed by the nonprofit peers network",
    "www.peerservice.org",
    "www.momentoflove.org",
    "www.personalgrowthcourses.net",
    "www.wanttoknow.info",
    "www.weboflove.org",
    "subscribe to the wanttoknow.info email list",
    "subscribe to the wanttoknow.info",
]

HEADER_MARKERS = [
    "wanttoknow.info inspiration center",
    "peers network",
    "momentoflove.org",
    "personalgrowthcourses.net",
    "weboflove.org",
]

def remove_footer_safe(text: str, tail_fraction: float = 0.4) -> str:
    """
    Remove footer content only if a marker appears in the *tail* of the document.
    tail_fraction = fraction of lines considered 'tail' (e.g., 0.4 => last 40%).
    """
    lines = text.splitlines()
    n = len(lines)
    if n == 0:
        return text

    # Start of the "footer tail" (only search from here to the end)
    tail_start_idx = int(n * (1.0 - tail_fraction))
    if tail_start_idx < 0:
        tail_start_idx = 0

    cut_line_idx = None

    for i in range(tail_start_idx, n):
        lower = lines[i].lower()
        for marker in FOOTER_MARKERS:
            if marker in lower:
                cut_line_idx = i
                break
        if cut_line_idx is not None:
            break

    if cut_line_idx is None:
        # No footer markers in the tail; leave text untouched
        return text.rstrip()

    # Keep everything *before* the footer marker line
    cleaned_lines = lines[:cut_line_idx]
    cleaned = "\n".join(cleaned_lines).rstrip()
    return cleaned

def remove_header(text: str, max_header_lines: int = 40) -> str:
    """
    Basic header cleanup:
    - Look at the first `max_header_lines` lines.
    - Drop lines that contain any HEADER_MARKERS.
    - Stop skipping once we see what looks like real content.
    """
    lines = text.splitlines()
    new_lines = []
    skipping = True

    for i, line in enumerate(lines):
        if i >= max_header_lines:
            # After this, keep everything
            new_lines.extend(lines[i:])
            break

        lower = line.lower().strip()

        if skipping:
            if not lower:
                # Skip leading blanks
                continue

            if any(m in lower for m in HEADER_MARKERS):
                # Drop obvious header boilerplate
                continue

            # Heuristic: if line looks like real content, stop skipping
            if "." in lower and len(lower) > 40:
                skipping = False
                new_lines.append(line)
            else:
                # Here we stop skipping as soon as line doesn't look like boilerplate.
                skipping = False
                new_lines.append(line)
        else:
            new_lines.append(line)

    if not new_lines:
        new_lines = lines

    cleaned = "\n".join(new_lines)
    return cleaned.strip()

# ---- APPLY CLEANUP TO REMAINING FILES ----

cleaned_count = 0

for fpath in kept_files:
    with fpath.open("r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    # 1) Header cleanup (conservative)
    text = remove_header(text)

    # 2) Footer cleanup, but only if markers appear in the last X% of lines
    text = remove_footer_safe(text, tail_fraction=0.4)  # last 40% scanned for footer

    # 3) Normalize extra blank lines
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    with fpath.open("w", encoding="utf-8") as f:
        f.write(text)

    cleaned_count += 1

print(f"Header/footer cleanup applied to {cleaned_count} files.")
